# Lesson 4.3 — Calibrating the Twin

Tell the twin about the effect it was missing: the gap closes for that effect. A new unmodeled effect opens a fresh gap.

In [1]:
import numpy as np
from modules.module09.integration import GreenhouseWorld, model_perception, understand
from modules.module10.twin import DigitalTwin, GroundTruth, outcome_gap, snapshot
def victim_and_xy(seed=1):
    w = GreenhouseWorld.demo_row(n=6, seed=seed)
    dets = model_perception(w, rng=np.random.default_rng(7))
    v = understand(dets, w)[1]['id']
    vxy = next(d['xy'] for d in dets if d['id']==v)
    return v, vxy
checks = []
v, vxy = victim_and_xy()
eff = {v: dict(obstacle=(vxy,0.25))}
ground = GroundTruth(GreenhouseWorld.demo_row(n=6, seed=1), unmodeled=eff)
twin = DigitalTwin(ground.world)
before = outcome_gap(twin.simulate(), ground.run())
print('before calibration -> match:', before['match'])
checks.append(before['match'] is False)
# calibrate: model the known effect
twin.calibrate(eff)
after = outcome_gap(twin.simulate(), ground.run())
print('after  calibration -> match:', after['match'])
checks.append(after['match'] is True)                  # gap closed for that effect
# a DIFFERENT unmodeled effect opens a new gap (calibration is per-effect)
v2, v2xy = victim_and_xy(seed=1)
others = [f.fid for f in ground.world.fruit if f.ripe and f.fid != v]
w = others[0]; dets = model_perception(ground.world, rng=np.random.default_rng(7))
wxy = next(d['xy'] for d in dets if d['id']==w)
ground2 = GroundTruth(GreenhouseWorld.demo_row(n=6, seed=1), unmodeled={**eff, w: dict(obstacle=(wxy,0.25))})
new_gap = outcome_gap(twin.simulate(), ground2.run())   # twin still only knows eff
print('new unmodeled effect on %s -> match:' % w, new_gap['match'])
checks.append(new_gap['match'] is False)               # residual remains
assert all(checks), f'FAILED: {checks}'
print(f'{sum(checks)}/{len(checks)} checks passed.')
print('All checks passed.')


before calibration -> match: False


after  calibration -> match: True


new unmodeled effect on F0 -> match: False
3/3 checks passed.
All checks passed.
